# Setup

In [1]:
!pip install evaluate --quiet

In [2]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g
...,...,...,...,...,...,...,...
2984,training_EN_2985,Lesbian magazine Curve 'saved people from lone...,A new documentary called Ahead of the Curve ch...,Somewhere in germany Kolon some guy has taken ...,no,no,none
2985,training_EN_2986,TURKEY: RIOT POLICE BREAK UP GAY PRIDE PARADE,"To License This Clip, Click Here: [URL]",Respect from Uzbekistan,no,yes_implicit,group_lgbtqia+
2986,training_EN_2987,Argentina’s transgender soccer star,"Meet Mara Gomez, Argentina’s first transgender...",Yeah.. super fair to all those REAL women.,yes,yes_implicit,individual_t
2987,training_EN_2988,Model comes out as intersex,Model Hanne Gaby Odiele has revealed that she ...,Good for her. Everyone has the right to be wan...,no,no,none


In [3]:
data['target_list'] = data.target.apply(lambda x: x.split(',') if x != 'none' else [])
data

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target,target_list
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l,[individual_l]
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t,[group_t]
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g","[group_l, g]"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l,[group_l]
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g,[group_g]
...,...,...,...,...,...,...,...,...
2984,training_EN_2985,Lesbian magazine Curve 'saved people from lone...,A new documentary called Ahead of the Curve ch...,Somewhere in germany Kolon some guy has taken ...,no,no,none,[]
2985,training_EN_2986,TURKEY: RIOT POLICE BREAK UP GAY PRIDE PARADE,"To License This Clip, Click Here: [URL]",Respect from Uzbekistan,no,yes_implicit,group_lgbtqia+,[group_lgbtqia+]
2986,training_EN_2987,Argentina’s transgender soccer star,"Meet Mara Gomez, Argentina’s first transgender...",Yeah.. super fair to all those REAL women.,yes,yes_implicit,individual_t,[individual_t]
2987,training_EN_2988,Model comes out as intersex,Model Hanne Gaby Odiele has revealed that she ...,Good for her. Everyone has the right to be wan...,no,no,none,[]


In [10]:
id2label = {idx: label for idx, label in enumerate(data.stereotype.unique())}
label2id = {label: idx for idx, label in id2label.items()}

In [11]:
data['label'] = data.stereotype.map(label2id)
data

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target,target_list,label
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l,[individual_l],0
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t,[group_t],0
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g","[group_l, g]",0
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l,[group_l],0
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g,[group_g],0
...,...,...,...,...,...,...,...,...,...
2984,training_EN_2985,Lesbian magazine Curve 'saved people from lone...,A new documentary called Ahead of the Curve ch...,Somewhere in germany Kolon some guy has taken ...,no,no,none,[],1
2985,training_EN_2986,TURKEY: RIOT POLICE BREAK UP GAY PRIDE PARADE,"To License This Clip, Click Here: [URL]",Respect from Uzbekistan,no,yes_implicit,group_lgbtqia+,[group_lgbtqia+],1
2986,training_EN_2987,Argentina’s transgender soccer star,"Meet Mara Gomez, Argentina’s first transgender...",Yeah.. super fair to all those REAL women.,yes,yes_implicit,individual_t,[individual_t],0
2987,training_EN_2988,Model comes out as intersex,Model Hanne Gaby Odiele has revealed that she ...,Good for her. Everyone has the right to be wan...,no,no,none,[],1


## Chia tập Train/Test dùng chung

Để so sánh công bằng giữa các phương pháp bên dưới (Bag-of-Words, Zero-shot, Sinh văn bản, Fine-tune Transformer), ta chia train/test **một lần duy nhất** ở đây và dùng lại cho mọi phương pháp. Minh hoạ trên **Subtask A — phát hiện stereotype** (nhị phân yes/no, cột `label`).

In [12]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data["label"]
)
print(f"Train: {len(train_df)} mẫu, Test: {len(test_df)} mẫu")
train_df["label"].value_counts(normalize=True)

Train: 2391 mẫu, Test: 598 mẫu


label
1    0.556253
0    0.443747
Name: proportion, dtype: float64

In [13]:
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Lưu kết quả của từng phương pháp để so sánh ở cuối notebook
results = {}

device = 0 if torch.cuda.is_available() else -1
print("Đang chạy trên:", "GPU" if device == 0 else "CPU")

ModuleNotFoundError: No module named 'torch'

## Phương pháp A: Bag-of-Words (TF-IDF) + Machine Learning truyền thống

Cách tiếp cận cổ điển trước khi có deep learning: biến mỗi câu bình luận thành một vector tần suất từ (Bag-of-Words / TF-IDF), rồi dùng các thuật toán học máy truyền thống (Logistic Regression, Naive Bayes, Linear SVM) để phân loại. Ưu điểm: không cần GPU, huấn luyện trong vài giây, dễ hiểu — là baseline nên so sánh trước khi dùng mô hình phức tạp hơn.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Biến văn bản thành vector TF-IDF (unigram + bigram)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
X_train = vectorizer.fit_transform(train_df["yt_comment"])
X_test = vectorizer.transform(test_df["yt_comment"])
y_train = train_df["label"]
y_test = test_df["label"]

bow_models = {
    "BoW + Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "BoW + Naive Bayes": MultinomialNB(),
    "BoW + Linear SVM": LinearSVC(class_weight="balanced"),
}

for name, clf in bow_models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    results[name] = {"accuracy": acc, "f1_macro": f1}
    print(f"--- {name} ---")
    print(classification_report(y_test, preds, target_names=[id2label[0], id2label[1]]))

## Phương pháp B: Zero-shot classification (không cần huấn luyện)

Thay vì huấn luyện, ta tận dụng một mô hình NLI (Natural Language Inference) đã huấn luyện sẵn để suy luận trực tiếp nhãn phù hợp nhất cho mỗi câu — không cần một dòng dữ liệu gán nhãn nào. Hữu ích khi mới bắt đầu, chưa có đủ dữ liệu train, hoặc muốn có một baseline nhanh để so sánh.

In [ ]:
from transformers import pipeline

zero_shot_clf = pipeline(
    "zero-shot-classification", model="facebook/bart-large-mnli", device=device
)

candidate_labels = [
    "a comment that expresses a stereotype about LGBTQIA+ people",
    "a comment that does not express a stereotype",
]

# Chỉ chạy trên một phần tập test cho nhanh (zero-shot chậm hơn nhiều so với BoW)
sample_test = test_df.sample(n=min(100, len(test_df)), random_state=42)

zs_preds = []
for comment in sample_test["yt_comment"]:
    out = zero_shot_clf(comment, candidate_labels)
    pred_label = 0 if out["labels"][0] == candidate_labels[0] else 1
    zs_preds.append(pred_label)

acc = accuracy_score(sample_test["label"], zs_preds)
f1 = f1_score(sample_test["label"], zs_preds, average="macro")
results["Zero-shot (BART-MNLI)"] = {"accuracy": acc, "f1_macro": f1}
print(f"Zero-shot accuracy: {acc:.3f}, F1-macro: {f1:.3f} (đánh giá trên {len(sample_test)} mẫu)")

## Phương pháp C: Sinh văn bản (Text-to-Text) với FLAN-T5

Khác với Phương pháp A, B, D vốn là bài toán *phân loại* (model chọn 1 trong N nhãn có sẵn), cách này biến phân loại thành bài toán *sinh văn bản*: đưa vào một câu lệnh (prompt) mô tả nhiệm vụ, model tự sinh ra chuỗi ký tự trả lời (ở đây là "yes" hoặc "no"). Đây là cách GPT, FLAN-T5 và nhiều LLM hiện đại giải quyết bài toán phân loại — không cần thêm classification head, và dễ mở rộng sang nhãn phức tạp hơn (ví dụ sinh thẳng cả loại hate speech) về sau.

In [ ]:
from transformers import AutoTokenizer as GenTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = GenTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
gen_model = gen_model.to("cuda" if device == 0 else "cpu")

def build_prompt(comment):
    return (
        "Does the following YouTube comment express a stereotype about "
        "LGBTQIA+ people? Answer only 'yes' or 'no'.\n"
        f"Comment: {comment}\nAnswer:"
    )

def generate_label(comment):
    prompt = build_prompt(comment)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True).to(gen_model.device)
    output_ids = gen_model.generate(**inputs, max_new_tokens=5)
    text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip().lower()
    return 0 if "yes" in text else 1  # 0 = yes (có stereotype), 1 = no - khớp với label2id

gen_preds = [generate_label(c) for c in sample_test["yt_comment"]]

acc = accuracy_score(sample_test["label"], gen_preds)
f1 = f1_score(sample_test["label"], gen_preds, average="macro")
results["Text-to-Text (FLAN-T5 zero-shot)"] = {"accuracy": acc, "f1_macro": f1}
print(f"FLAN-T5 generation accuracy: {acc:.3f}, F1-macro: {f1:.3f} (đánh giá trên {len(sample_test)} mẫu)")

## Phương pháp D: Fine-tune Transformer (XLM-RoBERTa)

Phương pháp mạnh nhất nhưng cũng tốn tài nguyên nhất: gắn thêm một lớp phân loại (classification head) lên trên một mô hình Transformer đã pretrain (XLM-RoBERTa), rồi huấn luyện lại toàn bộ (fine-tune) trên dữ liệu của Task 8. Cần GPU và vài phút huấn luyện, nhưng thường cho kết quả tốt hơn BoW và zero-shot vì model đã có sẵn hiểu biết ngôn ngữ sâu từ lúc pretrain.

### Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [27]:
tokens = tokenizer("Hello")
tokens

{'input_ids': [101, 7592, 102], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1]}

In [69]:
def preprocess(sample):
    tokens = tokenizer(sample['yt_comment'], truncation=True)
    return tokens

In [ ]:
from datasets import Dataset, DatasetDict

# Dùng lại đúng train_df/test_df đã chia ở phần "Chia tập Train/Test dùng chung"
# để mọi phương pháp đều được đánh giá trên cùng một tập test
raw_dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df[["yt_comment", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["yt_comment", "label"]], preserve_index=False),
})
raw_dataset_dict

In [ ]:
dataset_dict = raw_dataset_dict.map(preprocess, remove_columns=["yt_comment"])

In [73]:
dataset_dict    

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2391
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 598
    })
})

In [74]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Đánh giá cho bài toán multiclass

In [ ]:

import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    predictions = np.asarray(predictions).reshape(-1).astype(int)
    labels = np.asarray(labels).reshape(-1).astype(int)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "f1_macro": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["f1"],

        "precision_macro": precision_metric.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["precision"],

        "recall_macro": recall_metric.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["recall"]
    }


#### Đánh giá cho bài toán multiclass, multilabel

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # 1. Apply sigmoid for independent class probabilities
    probabilities = sigmoid(logits)

    # 2. Threshold probabilities at 0.5 to get binary predictions (0 or 1)
    predictions = (probabilities >= 0.5).astype(int)

    # 3. Compute metrics using binary predictions and binary label matrix
    return clf_metrics.compute(predictions=predictions, references=labels)



### Huấn luyện

#### Load model

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/xlm-roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Huấn luyện

In [ ]:
import torch
import gc
import numpy as np

# Force garbage collection and empty CUDA cache memory
gc.collect()
torch.cuda.empty_cache()

# Ensure the correct single-label compute_metrics is used
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    predictions = np.asarray(predictions).reshape(-1).astype(int)
    labels = np.asarray(labels).reshape(-1).astype(int)

    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"],
        "precision_macro": precision_metric.compute(predictions=predictions, references=labels, average="macro")["precision"],
        "recall_macro": recall_metric.compute(predictions=predictions, references=labels, average="macro")["recall"]
    }

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,  # Reduced to 2 to strictly fit within available GPU memory
    per_device_eval_batch_size=2,   # Reduced to 2
    gradient_accumulation_steps=8,  # Simulates batch size of 16 (2 * 8)
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="epoch",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.577538,0.685619,0.750000,0.679518,0.836795
2,No log,0.560384,0.693980,0.731278,0.723837,0.738872


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=300, training_loss=0.5584716288248698, metrics={'train_runtime': 157.0197, 'train_samples_per_second': 30.455, 'train_steps_per_second': 1.911, 'total_flos': 396797165921880.0, 'train_loss': 0.5584716288248698, 'epoch': 2.0})

In [ ]:
import numpy as np

pred_output = trainer.predict(dataset_dict["test"])
transformer_preds = np.argmax(pred_output.predictions, axis=-1)
transformer_labels = pred_output.label_ids

acc = accuracy_score(transformer_labels, transformer_preds)
f1 = f1_score(transformer_labels, transformer_preds, average="macro")
results["Fine-tune XLM-RoBERTa"] = {"accuracy": acc, "f1_macro": f1}
print(f"Transformer accuracy: {acc:.3f}, F1-macro: {f1:.3f}")

## So sánh các phương pháp

Tổng hợp kết quả của cả 4 hướng tiếp cận trên cùng một tập test (hoặc tập con của nó với B/C), để thấy rõ đánh đổi giữa độ phức tạp, thời gian chạy và độ chính xác.

In [ ]:
import pandas as pd

# Create summary comparison DataFrame and sort by F1-Macro score
summary = pd.DataFrame(results).T.sort_values("f1_macro", ascending=False)
summary

Lưu ý khi đọc bảng so sánh:

- **Phương pháp B (Zero-shot)** và **C (FLAN-T5)** chỉ đánh giá trên 100 mẫu con của tập test (để chạy nhanh), trong khi **A (BoW)** và **D (Transformer)** đánh giá trên toàn bộ tập test — muốn so sánh tuyệt đối công bằng thì nên chạy B/C trên toàn bộ tập test (sẽ chậm hơn nhiều).
- Đây là baseline minh hoạ cho **Subtask A (stereotype)**. Có thể lặp lại đúng quy trình này cho **Subtask B (hate speech, 3 lớp)** và **Subtask C (target, multilabel)** bằng cách đổi cột nhãn tương ứng và chỉnh lại số lớp (`num_labels`) hoặc dùng sigmoid/threshold cho multilabel (xem phần "Đánh giá cho bài toán multiclass, multilabel" ở trên).